## Indexer Ordering (Example: k = 2)

For each degree `k`, variables are stored in one contiguous block of size:

$2(k + 1)$

For `k = 2`, we have `m = 0,1,2`, so the block contains:

$s_{2,0} \quad s_{2,1}\quad s_{2,2} \quad i_{2,0} \quad i_{2,1} \quad i_{2,2}$ 

If this block starts at position `offsets[2]`, then:

- `idx_s(k,m) = offsets[k] + m`
- `idx_i(k,m) = offsets[k] + (k + 1) + m`

So for `k = 2`:

idx_s(2,0) → offsets[2] + 0 → $s_{2,0}$  
idx_s(2,1) → offsets[2] + 1 → $s_{2,1}$  
idx_s(2,2) → offsets[2] + 2 → $s_{2,2}$  
idx_i(2,0) → offsets[2] + 3 → $i_{2,0}$  
idx_i(2,1) → offsets[2] + 4 → $i_{2,1}$  
idx_i(2,2) → offsets[2] + 5 → $i_{2,2}$  

The first `k+1` entries in the block are all `s`,  
the next `k+1` entries are all `i`.


In [1]:
#Declare object
struct Indexer
    k_vals::Vector{Int}
    offsets::Dict{Int,Int}
    total::Int
end

function build_indexer(k_vals) #Recives a list of node degrees
    offsets = Dict{Int,Int}()
    counter = 1
    for k in k_vals #Build offsets
        offsets[k] = counter
        counter += 2*(k+1) 
    end
    return Indexer(k_vals, offsets, counter-1)
end

function idx_s(ind::Indexer, k, m)
    return ind.offsets[k] + m
end

function idx_i(ind::Indexer, k, m)
    return ind.offsets[k] + (k+1) + m
end


idx_i (generic function with 1 method)

Note that `k_vals` goes from $k_\text{min}$ to $k_\text{max}$ so `Pk[]` has dimensions of $k_\text{max} - k_\text{min} + 1$

In [2]:
function compute_rates!(x, ind, Pk)
    """
    In:
     - x: Variables
     - ind: Indexer
     - Pk: List of degree distribution (quenched or annealed)
    
    """

    βs_num = 0.0
    βs_den = 0.0
    γs_num = 0.0
    γs_den = 0.0

    βi_num = 0.0
    βi_den = 0.0
    γi_num = 0.0
    γi_den = 0.0

    for k in ind.k_vals
        P = Pk[k] #
        for m in 0:k
            s = x[idx_s(ind,k,m)]
            i = x[idx_i(ind,k,m)]

            βs_num += P * (m*(k-m)/k) * s
            βs_den += P * (k-m) * s

            γs_num += P * ((k-m)^2/k) * i
            γs_den += P * (k-m) * i

            βi_num += P * (m^2/k) * s
            βi_den += P * m * s

            γi_num += P * (m*(k-m)/k) * i
            γi_den += P * m * i
        end
    end

    βs = βs_num / βs_den
    γs = γs_num / γs_den
    βi = βi_num / βi_den
    γi = γi_num / γi_den

    return βs, γs, βi, γi
end

compute_rates! (generic function with 1 method)

In [3]:
function initial_guess(ind, ρ0)

    x0 = zeros(ind.total)

    for k in ind.k_vals
        for m in 0:k
            B = binomial(k,m) * ρ0^m * (1-ρ0)^(k-m)

            x0[idx_s(ind,k,m)] = (1-ρ0)*B
            x0[idx_i(ind,k,m)] = ρ0*B
        end
    end

    return x0
end



initial_guess (generic function with 1 method)

# Non-Linear System

In [7]:
function residual!(F, x, ind, Pk)

    βs, γs, βi, γi = compute_rates!(x, ind, Pk)

    for k in ind.k_vals
        for m in 0:k

            s_idx = idx_s(ind,k,m)
            i_idx = idx_i(ind,k,m)

            s = x[s_idx]
            i = x[i_idx]

            # neighbors applying boundary conditions
            s_m1 = m>0  ? x[idx_s(ind,k,m-1)] : 0.0
            s_p1 = m<k  ? x[idx_s(ind,k,m+1)] : 0.0

            i_m1 = m>0  ? x[idx_i(ind,k,m-1)] : 0.0
            i_p1 = m<k  ? x[idx_i(ind,k,m+1)] : 0.0

            F[s_idx] =
                -(m/k)*s + ((k-m)/k)*i
                - βs*(k-m)*s
                + βs*(k-m+1)*s_m1
                - γs*m*s
                + γs*(m+1)*s_p1

            F[i_idx] =
                -((k-m)/k)*i + (m/k)*s
                - βi*(k-m)*i
                + βi*(k-m+1)*i_m1
                - γi*m*i
                + γi*(m+1)*i_p1
        end
    end

    return nothing
end


residual! (generic function with 1 method)

In [8]:
using Graphs  
using StatsBase 

N = 500      # number of nodes
p = 0.01      # probability of edge between any two nodes

# g = erdos_renyi(N, p)
g = random_regular_graph(N, 4)

deg = degree.(Ref(g), 1:N)

max_k = maximum(deg)           # largest observed degree
min_k = minimum(deg)
k_vals = collect(min_k:max_k)
counts = countmap(deg)         # counts of each degree
total_nodes = nv(g)

# Convert counts to probabilities
Pk0 = Dict(k => v/total_nodes for (k,v) in counts)
Pk = Dict(k => get(Pk0, k, 0.0) for k in 0:max_k)

Dict{Int64, Float64} with 5 entries:
  0 => 0.0
  4 => 1.0
  2 => 0.0
  3 => 0.0
  1 => 0.0

In [9]:
using Optim

# --- Setup indexer and initial guess ---
ind = build_indexer(k_vals)
x0 = initial_guess(ind, 0.5)  # your initial guess vector

# --- Define a residual norm function ---
function residual_norm(x)
    F = similar(x)
    residual!(F, x, ind, Pk)   # compute your residual vector
    return sum(abs2, F)        # squared L2 norm of residual
end

# --- Set up bounds: all variables between 0 and 1 ---
lower = zeros(length(x0))
upper = ones(length(x0))

# --- Optimize ---
result = optimize(residual_norm, lower, upper, x0, Fminbox(), Optim.Options(f_abstol=1e-20))

# --- Extract solution ---
x_sol = Optim.minimizer(result)

# --- Print some info ---
println("Residual norm: ", residual_norm(x_sol))
# println("Solution vector: ", x_sol)

# Example: print s_{2,1} using the indexer
k = 4
m = 2
println("s_{4,2} = ", x_sol[idx_s(ind, k, m)])



Residual norm: 1.2408976451151496e-17
s_{2,1} = 0.5000000081763668


In [10]:
using NLsolve

ind = build_indexer(k_vals)
x0 = initial_guess(ind, 0.5) 
f!(F, x) = residual!(F, x, ind, Pk) 
sol = nlsolve(f!, x0; method=:trust_region, xtol=1e-20) # Check if the residual converged 

println("Residual converged? ", sol.f_converged) # Print the full solution vector 
println("Solution vector: ", sol.zero) # Print a specific variable, e.g., s_{2,1} using the indexer k = 2 m = 1
# println("s_{2,1} = ", sol.zero[idx_s(ind, k, m)])

Residual converged? true
Solution vector: [0.03125, 0.1500000000000987, 0.1875, 0.04999999999999041, -6.9735883734267645e-15, -6.9735883734267645e-15, 0.04999999999999041, 0.1875, 0.1500000000000987, 0.03125]


In [11]:
for rho0 in 0.0:0.1:1.0
    x0 = initial_guess(ind, rho0)
    sol = nlsolve(f!, x0; method=:trust_region, xtol=1e-20)
    println("rho0 = $rho0, s_{4,2} = ", sol.zero[idx_s(ind,4,2)])
end

rho0 = 0.0, s_{4,2} = 0.0
rho0 = 0.1, s_{4,2} = 0.02430000000000123
rho0 = 0.2, s_{4,2} = 0.07680000000002933
rho0 = 0.3, s_{4,2} = 0.1323000000000336
rho0 = 0.4, s_{4,2} = 0.172800000000022
rho0 = 0.5, s_{4,2} = 0.1875
rho0 = 0.6, s_{4,2} = 0.17279999999997808
rho0 = 0.7, s_{4,2} = 0.13229999999996636
rho0 = 0.8, s_{4,2} = 0.07679999999997068
rho0 = 0.9, s_{4,2} = 0.024300000000009897
rho0 = 1.0, s_{4,2} = 0.0


# Differential Equation

In [15]:
using DifferentialEquations

function ode_system!(dx, x, p, t)

    ind, Pk, k_vals = p

    fill!(dx, 0.0)

    # Compute the global transition rates from current state
    βs, βi, γs, γi = compute_rates!(x, ind, Pk)

    for k in k_vals
        for m in 0:k

            is = idx_s(ind,k,m)
            ii = idx_i(ind,k,m)

            s = x[is]
            i = x[ii]

            # Handle boundaries
            s_mminus = m > 0 ? x[idx_s(ind,k,m-1)] : 0.0
            s_mplus  = m < k ? x[idx_s(ind,k,m+1)] : 0.0

            i_mminus = m > 0 ? x[idx_i(ind,k,m-1)] : 0.0
            i_mplus  = m < k ? x[idx_i(ind,k,m+1)] : 0.0

            # ds_{k,m}/dt
            dx[is] =
                -(m/k)*s +
                (k-m)/k * i
                - βs*(k-m)*s
                + βs*(k-m+1)*s_mminus
                - γs*m*s
                + γs*(m+1)*s_mplus

            # di_{k,m}/dt
            dx[ii] =
                -(k-m)/k * i +
                (m/k)*s
                - βi*(k-m)*i
                + βi*(k-m+1)*i_mminus
                - γi*m*i
                + γi*(m+1)*i_mplus

        end
    end

end


ode_system! (generic function with 1 method)

In [22]:
using Graphs  
using StatsBase 

N = 100      # number of nodes
p = 0.01      # probability of edge between any two nodes

# g = erdos_renyi(N, p)
g = random_regular_graph(N, 4)

deg = degree.(Ref(g), 1:N)

max_k = maximum(deg)           # largest observed degree
min_k = minimum(deg)
k_vals = collect(min_k:max_k)
counts = countmap(deg)         # counts of each degree
total_nodes = nv(g)

# Convert counts to probabilities
Pk0 = Dict(k => v/total_nodes for (k,v) in counts)
Pk = Dict(k => get(Pk0, k, 0.0) for k in 0:max_k)

Dict{Int64, Float64} with 5 entries:
  0 => 0.0
  4 => 1.0
  2 => 0.0
  3 => 0.0
  1 => 0.0

In [23]:
function density(x, ind, k_vals)
    total = 0.0
    for k in k_vals, m in 0:k
        total += x[idx_i(ind,k,m)]
    end
    total
end


density (generic function with 1 method)

In [24]:
using DifferentialEquations

ind = build_indexer(k_vals)

p = (ind, Pk, k_vals)
tspan = (0.0,10000.0)

k = 4
m = 2

for rho0 in 0.0:0.1:1.0
    x0 = initial_guess(ind, rho0)
    
    prob = ODEProblem(ode_system!, x0, tspan, p)
    
    sol = solve(prob, Tsit5(), reltol=1e-8, abstol=1e-8)
    
    
    x_final = sol[end]    
    println("rho0 = $rho0, s_{4,2} = ", x_final[idx_s(ind,k,m)])
    # println("i_{4,1} = ", x_final[idx_i(ind,k,m)])
end

rho0 = 0.0, s_{4,2} = 0.0
rho0 = 0.1, s_{4,2} = 0.02430000055249771
rho0 = 0.2, s_{4,2} = 0.07680000066958757
rho0 = 0.3, s_{4,2} = 0.13230000330713892
rho0 = 0.4, s_{4,2} = 0.17280000193097503
rho0 = 0.5, s_{4,2} = 0.1875
rho0 = 0.6, s_{4,2} = 0.17279999806902505
rho0 = 0.7, s_{4,2} = 0.1322999966928611
rho0 = 0.8, s_{4,2} = 0.07679999933041515
rho0 = 0.9, s_{4,2} = 0.024299999447502794
rho0 = 1.0, s_{4,2} = 0.0
